# 02 — Denial EDA & Quarterly Rate Reproduction

Exploratory analysis of the synthetic denial log, plus reproduction of quarterly denial rates **8.2% / 9.6% / 11.7%** from `data/processed/quarterly_claims_summary.csv`.

**Claim-level data is synthetic**, CMS-calibrated. Dollar figures below use the same formulae as `02_denial_analysis/sql/05_revenue_at_risk.sql`.


In [1]:
from pathlib import Path
import pandas as pd

ROOT = Path.cwd().resolve()
for candidate in [ROOT, ROOT.parent, ROOT.parent.parent]:
    if (candidate / 'data' / 'processed' / 'quarterly_claims_summary.csv').exists():
        ROOT = candidate
        break

processed = ROOT / 'data' / 'processed'
quarterly = pd.read_csv(processed / 'quarterly_claims_summary.csv')
constants = pd.read_csv(processed / 'analysis_constants.csv')
denials = pd.read_csv(processed / 'riverside_denial_log.csv')
quarterly


,Quarter,Claims_Submitted,Claims_Denied,Denial_Rate,Denial_Rate_Pct,Avg_Allowed_Amount,Revenue_at_Risk
0,Q3 2024,168000,13776,0.082,8.2,79.73,1098360.48
1,Q4 2024,172000,16512,0.096,9.6,79.73,1316501.76
2,Q1 2025,171000,20007,0.117,11.7,79.73,1595158.11


## Quarterly denial rates (target: 8.2%, 9.6%, 11.7%)

In [2]:
rates = quarterly.copy()
rates['Computed_Denial_Rate_Pct'] = (
    100.0 * rates['Claims_Denied'] / rates['Claims_Submitted']
).round(1)
display_cols = ['Quarter', 'Claims_Submitted', 'Claims_Denied', 'Computed_Denial_Rate_Pct']
rates[display_cols]


,Quarter,Claims_Submitted,Claims_Denied,Computed_Denial_Rate_Pct
0,Q3 2024,168000,13776,8.2
1,Q4 2024,172000,16512,9.6
2,Q1 2025,171000,20007,11.7


In [3]:
expected = {'Q3 2024': 8.2, 'Q4 2024': 9.6, 'Q1 2025': 11.7}
computed = dict(zip(rates['Quarter'], rates['Computed_Denial_Rate_Pct']))
assert computed == expected, f'Rate mismatch: {computed} != {expected}'
print('Quarterly denial rates reproduced:')
for q, r in computed.items():
    print(f'  {q}: {r:.1f}%')


Quarterly denial rates reproduced:
  Q3 2024: 8.2%
  Q4 2024: 9.6%
  Q1 2025: 11.7%


## Revenue at risk (ties to `05_revenue_at_risk.sql`)

In [4]:
rates['Computed_Revenue_at_Risk'] = (
    rates['Claims_Denied'] * rates['Avg_Allowed_Amount']
).round(2)
rates[['Quarter', 'Avg_Allowed_Amount', 'Computed_Revenue_at_Risk', 'Revenue_at_Risk']]


,Quarter,Avg_Allowed_Amount,Computed_Revenue_at_Risk,Revenue_at_Risk
0,Q3 2024,79.73,1098360.48,1098360.48
1,Q4 2024,79.73,1316501.76,1316501.76
2,Q1 2025,79.73,1595158.11,1595158.11


In [5]:
consts = dict(zip(constants['Constant'], constants['Value']))
incremental = round(
    consts['ANNUAL_CLAIMS']
    * (consts['LATEST_DENIAL_RATE'] - consts['BASELINE_DENIAL_RATE'])
    * consts['AVG_ALLOWED_AMOUNT'],
    2,
)
print(f"Incremental annual revenue exposure: ${incremental:,.2f}")
assert incremental == consts['INCREMENTAL_ANNUAL_REVENUE_EXPOSURE']
print('Headline figure reconciled to analysis_constants.csv')


Incremental annual revenue exposure: $1,897,574.00
Headline figure reconciled to analysis_constants.csv


## Top reason codes (denial log sample)

In [6]:
mix = (
    denials.groupby(['Denial_Code', 'Denial_Description'])
    .size()
    .reset_index(name='Total_Claims')
    .sort_values('Total_Claims', ascending=False)
)
mix['Share_Pct'] = (100.0 * mix['Total_Claims'] / mix['Total_Claims'].sum()).round(1)
mix


,Denial_Code,Denial_Description,Total_Claims,Share_Pct
2,CO-4,Inconsistent modifier / procedure code,140,28.0
0,CO-16,Missing or incomplete claim information,120,24.0
7,PR-96,Non-covered charge / payer ID mismatch,95,19.0
1,CO-29,The time limit for filing has expired,40,8.0
3,CO-50,Non-covered services — not deemed medically ne...,35,7.0
4,CO-97,Benefit for this service is included in paymen...,30,6.0
6,PR-1,Deductible amount,25,5.0
5,OA-23,Impact of prior payer adjudication,15,3.0


In [7]:
q1_rar = float(quarterly.loc[quarterly['Quarter']=='Q1 2025', 'Revenue_at_Risk'].iloc[0])
top3 = mix[mix['Denial_Code'].isin(['CO-4','CO-16','PR-96'])].copy()
top3['Est_Impact'] = (q1_rar * top3['Share_Pct'] / 100.0).round(2)
top3[['Denial_Code', 'Share_Pct', 'Est_Impact']]


,Denial_Code,Share_Pct,Est_Impact
2,CO-4,28.0,446644.27
0,CO-16,24.0,382837.95
7,PR-96,19.0,303080.04


## Clinic × denial code (top 3)

In [8]:
(
    denials[denials['Denial_Code'].isin(['CO-4','CO-16','PR-96'])]
    .groupby(['Clinic_Location', 'Denial_Code'])
    .size()
    .reset_index(name='Total_Claims')
    .sort_values(['Clinic_Location', 'Total_Claims'], ascending=[True, False])
)


,Clinic_Location,Denial_Code,Total_Claims
0,Brookfield,CO-16,18
1,Brookfield,CO-4,15
2,Brookfield,PR-96,12
3,Downtown Milwaukee,CO-16,40
4,Downtown Milwaukee,CO-4,39
5,Downtown Milwaukee,PR-96,35
6,Southside,CO-16,18
8,Southside,PR-96,15
7,Southside,CO-4,12
10,Wauwatosa,CO-4,20
